# **Project Name** - Amazon Prime Video Catalog Analysis

##### **Project Type**    - EDA/Regression/Classification/Unsupervised
**Selected project type** - EDA
##### **Contribution**    - Individual/Team
**Selected contribution** - Individual
##### **Name -** Vishesh Rajak

# **Project Summary -**

This project examines the supplied Amazon Prime Video titles and credits snapshot to describe catalog composition, release history, and public audience-rating patterns. The project brief describes US catalog availability. The titles file contains 9,871 rows and 15 metadata fields; the credits file contains 124,235 actor and director credits. Title metadata includes format, release year, runtime, certification, genre and production-country lists, IMDb scores and votes, and TMDB measures. Credits link people and roles to title IDs. These are catalog records and public ratings, not customer viewing data.

The analysis begins with a reproducible audit of row and column counts, types, duplicates, missing values, numeric summaries, and distinct values. Three repeated title IDs and 56 exact duplicate credit rows are removed. Genre and country strings are parsed safely into lists. Movie season counts are structurally not applicable; show seasons remain available for show-only comparisons. Missing certification is labeled Unknown and missing descriptions are retained as empty text for text processing. Ratings are not fabricated: unrated titles remain in catalog counts but are excluded from rating comparisons. The machine-learning notebook imputes numeric predictors only inside each training fold.

The cleaned table contains 9,868 unique title IDs: 8,514 movies and 1,357 shows. Releases span 1912 to 2022, with the largest volume in the 2010s. The 2020s cover only the first three years in this file, so their count should not be compared with a complete decade. There are 8,850 observed IMDb scores with a mean near 5.98 and median of 6.1. Rated shows average 7.12 and movies 5.80. This difference is descriptive and may reflect format, genre mix, rating coverage, or selection rather than a causal effect. Drama, comedy, and thriller are common tags. Genre and country lists are multi-label, so counts overlap. Documentation titles have a relatively high average score among groups with enough observations.

The analysis follows a univariate, bivariate, and multivariate sequence. It compares format, release periods, runtime, show seasons, certification, genres, countries, ratings, and credited talent. Country and talent score comparisons use minimum sample thresholds to reduce unstable rankings. Each chart is paired with a reason for the visual choice, the observed pattern, and a practical interpretation. The plots also show missingness and numeric relationships, so limitations are visible beside the substantive findings.

The findings suggest several next steps. Review the movie and series balance against actual acquisition cost, viewing hours, and completion rates. Use genre and country results to form audience experiments rather than as standalone greenlight rules. Improve certification coverage before relying on age labels for family-suitability planning. Weight public ratings by vote count and report rated-title coverage with each average. Add rights cost, title-level consumption, release-window, and subscriber-retention data before estimating return on content investment. The current snapshot has no spend, rights, user, or retention measures, and external ratings are not a substitute for those outcomes. Observed relationships are descriptive and should guide tests with richer first-party data.

Group comparisons show rated-title counts and apply minimum thresholds to smaller country and talent groups. Multi-genre and production-country tags are not mutually exclusive. These choices make the summaries easier to interpret, but they do not remove selection effects or missing-data bias.

# **GitHub Link -**

```markdown
**GitHub Link:** https://github.com/VisheshRajak/Amazon_Prime_Video_Project_Ds-Aiml

# **Problem Statement**


Amazon Prime Video must decide how to balance a large catalog of films and series across genres and production markets while maintaining audience satisfaction. The supplied snapshot contains title metadata and public ratings, but not production or licensing cost, viewing hours, subscriber retention, or title availability dates. Without a structured audit, teams may overvalue catalog volume, compare formats with different runtime meanings, or draw conclusions from small groups and incomplete metadata. This analysis describes catalog composition, rating patterns, release trends, and credited talent while separating associations from business outcomes the files cannot measure.

#### **Define Your Business Objective?**

Describe the catalog by format, release period, genre, certification, production market, and credited talent; compare public ratings across groups with sample-size safeguards; identify data-quality limits; and translate findings into testable acquisition and metadata recommendations. This project does not estimate ROI or causality because the supplied data contain no cost, viewing, or retention measures.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

import ast
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', category=FutureWarning)

### Dataset Loading

In [ ]:
# Load Dataset

def _find_dataset_file(filename):
    candidates = [
        Path('/content') / filename,
        Path.cwd() / filename,
        Path.cwd() / 'data' / filename,
        Path.home() / 'Downloads' / filename,
        Path.home() / 'Downloads' / filename.replace('.csv', '') / filename,
    ]
    return next((path for path in candidates if path.is_file()), None)

titles_path = _find_dataset_file('titles.csv')
credits_path = _find_dataset_file('credits.csv')
if titles_path is None or credits_path is None:
    try:
        from google.colab import files
        print('Upload both titles.csv and credits.csv to continue.')
        uploaded = files.upload()
        titles_path = titles_path or next((Path(name) for name in uploaded if Path(name).name == 'titles.csv'), None)
        credits_path = credits_path or next((Path(name) for name in uploaded if Path(name).name == 'credits.csv'), None)
    except ImportError:
        pass
if titles_path is None or credits_path is None:
    raise FileNotFoundError('Place titles.csv and credits.csv in /content, the notebook folder, or its data/ folder.')

titles_raw_df = pd.read_csv(titles_path)
credits_raw_df = pd.read_csv(credits_path)
print(f'Titles loaded from {titles_path}')
print(f'Credits loaded from {credits_path}')

### Dataset First View

In [ ]:
# Dataset First Look

display(titles_raw_df.head())
display(credits_raw_df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count

print('Titles shape:',titles_raw_df.shape)
print('Credits shape:',credits_raw_df.shape)

### Dataset Information

In [ ]:
# Dataset Info

titles_raw_df.info()
print('\nCredits data types:')
display(credits_raw_df.dtypes.to_frame('dtype'))

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count

print('Duplicate title IDs:',titles_raw_df['id'].duplicated().sum())
print('Duplicate credit rows:',credits_raw_df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count

display(titles_raw_df.isna().sum().sort_values(ascending=False).to_frame('missing_rows'))
print('Credit missing values:')
display(credits_raw_df.isna().sum().to_frame('missing_rows'))

In [ ]:
# Visualizing the missing values

missing_pct=titles_raw_df.isna().mean().mul(100).sort_values(ascending=False)
fig,ax=plt.subplots(figsize=(10,4)); missing_pct[missing_pct.gt(0)].plot(kind='bar',ax=ax,color='#F58518')
ax.set(title='Missing-value share in original title data',xlabel='Field',ylabel='Rows missing (%)')
ax.tick_params(axis='x',rotation=45); fig.tight_layout(); plt.show()

### What did you know about your dataset?

The cleaned title table has 9,868 unique title IDs after removing three repeated IDs. It covers 8,514 movies and 1,357 shows, with release years from 1912 through 2022. There are 8,850 non-missing IMDb scores. Movie rows dominate. Show season counts are structurally absent for movies, so season analysis is limited to shows. The credits table has 124,179 distinct rows after removing 56 exact duplicates and includes actor and director roles. Genre and production-country lists can contain multiple values per title, so exploded counts overlap. The project brief describes US catalog availability. The files do not provide user viewing, costs, licensing terms, retention, or time-stamped availability history.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns

print("Titles Columns:", list(titles_df.columns))
print("Credits Columns:", list(credits_df.columns))

In [ ]:
# Dataset Describe
display(titles_raw_df.select_dtypes(include='number').describe().T.round(2))

### Variables Description

| Field | Meaning | Analysis use |
|---|---|---|
| id, title, imdb_id | Title identifiers and name | Join and audit; excluded from prediction |
| type | MOVIE or SHOW | Format comparison and categorical predictor |
| description | Short synopsis | Text exploration and TF-IDF feature |
| release_year | Release year | Trend analysis and numeric predictor |
| age_certification | Audience certification label | Coverage audit and categorical predictor |
| runtime | Runtime in minutes | Numeric comparison; show runtime is episode-level |
| genres | Multi-label genre list | Genre analysis and text-vectorized predictor |
| production_countries | Multi-label country codes | Market mix and text-vectorized predictor |
| seasons | Number of seasons for shows | Show-only analysis; structurally absent for movies |
| imdb_score, imdb_votes | Public score and vote count | Outcome and description; votes excluded from pre-release model |
| tmdb_popularity, tmdb_score | Public TMDB measures | Descriptive analysis; excluded from pre-release model |
| person_id, name, character, role | Credited person, character and role | Talent summary after joining on title id |

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.

display(pd.DataFrame({'column':titles_raw_df.columns,'unique_values':[titles_raw_df[c].nunique(dropna=True) for c in titles_raw_df.columns]}))

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

def safe_parse_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed=ast.literal_eval(value) if isinstance(value,str) else value
        return parsed if isinstance(parsed,list) else []
    except (ValueError,SyntaxError,TypeError):
        return []

titles_df=titles_raw_df.drop_duplicates(subset='id',keep='first').copy()
credits_df=credits_raw_df.drop_duplicates().copy()
for col in ['runtime','release_year','seasons','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']:
    titles_df[col]=pd.to_numeric(titles_df[col],errors='coerce')
titles_df['genres_list']=titles_df['genres'].map(safe_parse_list)
titles_df['production_countries_list']=titles_df['production_countries'].map(safe_parse_list)
titles_df['age_certification']=titles_df['age_certification'].fillna('Unknown').replace('','Unknown')
titles_df['description']=titles_df['description'].fillna('').astype(str)
show_median_seasons=titles_df.loc[titles_df['type'].eq('SHOW'),'seasons'].median()
titles_df['seasons_for_model']=titles_df['seasons'].fillna(0)
titles_df.loc[titles_df['type'].eq('SHOW'),'seasons_for_model']=titles_df.loc[titles_df['type'].eq('SHOW'),'seasons'].fillna(show_median_seasons)
titles_df['decade']=(titles_df['release_year']//10*10).astype('Int64')
print('Titles after deduplication:',titles_df.shape)
print('Credit rows after exact deduplication:',credits_df.shape)
print('Remaining missing IMDb scores:',titles_df['imdb_score'].isna().sum())

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

# Ensure titles_raw_df and credits_raw_df are loaded if not already present
if 'titles_raw_df' not in globals() or 'credits_raw_df' not in globals():
    def _find_dataset_file(filename):
        candidates = [
            Path('/content') / filename,
            Path.cwd() / filename,
            Path.cwd() / 'data' / filename,
        ]
        return next((path for path in candidates if path.is_file()), None)

    titles_path = _find_dataset_file('titles.csv')
    credits_path = _find_dataset_file('credits.csv')

    if titles_path and credits_path:
        titles_raw_df = pd.read_csv(titles_path)
        credits_raw_df = pd.read_csv(credits_path)
    else:
        raise FileNotFoundError("Please run the Dataset Loading cell first to provide titles.csv and credits.csv.")

# Safely parse list representations from the CSV columns
def safe_parse_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = ast.literal_eval(value) if isinstance(value, str) else value
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []

# Deduplicate and clean titles data
titles_df = titles_raw_df.drop_duplicates(subset='id', keep='first').copy()
credits_df = credits_raw_df.drop_duplicates().copy()

for col in ['runtime', 'release_year', 'seasons', 'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score']:
    titles_df[col] = pd.to_numeric(titles_df[col], errors='coerce')

titles_df['genres_list'] = titles_df['genres'].map(safe_parse_list)
titles_df['production_countries_list'] = titles_df['production_countries'].map(safe_parse_list)
titles_df['age_certification'] = titles_df['age_certification'].fillna('Unknown').replace('', 'Unknown')
titles_df['description'] = titles_df['description'].fillna('').astype(str)

# Setup seasons for modeling
show_median_seasons = titles_df.loc[titles_df['type'].eq('SHOW'), 'seasons'].median()
titles_df['seasons_for_model'] = titles_df['seasons'].fillna(0)
titles_df.loc[titles_df['type'].eq('SHOW'), 'seasons_for_model'] = titles_df.loc[titles_df['type'].eq('SHOW'), 'seasons'].fillna(show_median_seasons)
titles_df['decade'] = (titles_df['release_year'] // 10 * 10).astype('Int64')

print("Data wrangling successfully executed.")
print(f"Cleaned Titles count: {len(titles_df)}")
print(f"Cleaned Credits count: {len(credits_df)}")

### What all manipulations have you done and insights you found?

Removed three duplicate title IDs and 56 exact duplicate credit rows. Parsed genre and production-country strings with a safe list parser. Converted numeric fields and added decade and list-count features for charts. Labeled missing certification as Unknown and filled missing descriptions with empty text only for text processing. Movie season counts are structurally not applicable; show seasons are summarized among shows only. Rating charts exclude missing IMDb scores. Talent ratings are joined by title ID and restricted to people with at least ten credited titles to reduce single-title noise.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code

import matplotlib.pyplot as plt
import seaborn as sns

def plot_catalog_chart(number):
    """Draw a reproducible chart from the cleaned title and credit tables."""
    import matplotlib.pyplot as plt
    import seaborn as sns
    if number == 1:
        counts=titles_df['type'].value_counts().reindex(['MOVIE','SHOW']).dropna()
        fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(counts.index,counts.values,color=['#4C78A8','#F58518'])
        ax.bar_label(bars,fmt='{:,.0f}'); ax.set(title='Catalog titles by format',xlabel='Format',ylabel='Titles')
    elif number == 2:
        fig,ax=plt.subplots(figsize=(10,4)); sns.histplot(data=titles_df,x='release_year',hue='type',multiple='stack',binwidth=5,ax=ax)
        ax.set(title='Release-year distribution in five-year bins',xlabel='Release year',ylabel='Titles')
    elif number == 3:
        fig,ax=plt.subplots(figsize=(8,4)); sns.boxplot(data=titles_df,x='type',y='runtime',showfliers=False,ax=ax)
        ax.set(title='Runtime differs by format',xlabel='Format',ylabel='Runtime (minutes)')
    elif number == 4:
        d=titles_df.dropna(subset=['imdb_score']); fig,ax=plt.subplots(figsize=(8,4))
        sns.histplot(data=d,x='imdb_score',bins=25,kde=True,color='#4C78A8',ax=ax)
        ax.set(title='IMDb score distribution',xlabel='IMDb score (0–10)',ylabel='Rated titles')
    elif number == 5:
        d=titles_df.loc[titles_df['type'].eq('SHOW')].dropna(subset=['seasons'])
        fig,ax=plt.subplots(figsize=(8,4)); sns.histplot(data=d,x='seasons',discrete=True,color='#F58518',ax=ax)
        ax.set(title='Season counts for shows',xlabel='Number of seasons',ylabel='Shows')
    elif number == 6:
        d=titles_df.dropna(subset=['imdb_score']); fig,ax=plt.subplots(figsize=(8,4))
        sns.boxplot(data=d,x='type',y='imdb_score',showfliers=False,ax=ax)
        ax.set(title='IMDb scores by format',xlabel='Format',ylabel='IMDb score')
    elif number == 7:
        d=titles_df.groupby(['release_year','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,4)); sns.lineplot(data=d,x='release_year',y='titles',hue='type',estimator=None,ax=ax)
        ax.set(title='Annual catalog volume by format',xlabel='Release year',ylabel='Titles in snapshot')
    elif number == 8:
        d=titles_df.assign(decade=titles_df['release_year']//10*10).groupby(['decade','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,4)); sns.barplot(data=d,x='decade',y='titles',hue='type',ax=ax)
        ax.tick_params(axis='x',rotation=45); ax.set(title='Catalog volume by decade and format',xlabel='Release decade',ylabel='Titles')
    elif number == 9:
        d=titles_df.dropna(subset=['imdb_score']).assign(decade=lambda x:x['release_year']//10*10).groupby('decade',as_index=False)['imdb_score'].mean()
        fig,ax=plt.subplots(figsize=(9,4)); sns.lineplot(data=d,x='decade',y='imdb_score',marker='o',ax=ax)
        ax.set(title='Mean IMDb score by release decade',xlabel='Release decade',ylabel='Mean IMDb score',ylim=(0,10))
    elif number == 10:
        d=titles_df.dropna(subset=['runtime','imdb_score']); d=d.sample(min(3000,len(d)),random_state=42)
        fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='runtime',y='imdb_score',hue='type',alpha=.35,s=22,ax=ax)
        ax.set(title='Runtime and IMDb score',xlabel='Runtime (minutes)',ylabel='IMDb score')
    elif number == 11:
        d=titles_df.dropna(subset=['imdb_votes','imdb_score']).sample(frac=1,random_state=42)
        d=d.head(min(3000,len(d))); fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='imdb_votes',y='imdb_score',alpha=.35,s=22,ax=ax)
        ax.set_xscale('log'); ax.set(title='IMDb votes and score',xlabel='IMDb votes (log scale)',ylabel='IMDb score')
    elif number == 12:
        d=titles_df.dropna(subset=['tmdb_popularity','imdb_score']).sample(frac=1,random_state=42)
        d=d.head(min(3000,len(d))); fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='tmdb_popularity',y='imdb_score',alpha=.35,s=22,ax=ax)
        ax.set_xscale('log'); ax.set(title='TMDB popularity and IMDb score',xlabel='TMDB popularity (log scale)',ylabel='IMDb score')
    elif number == 13:
        d=titles_df[['genres_list']].explode('genres_list').dropna(subset=['genres_list'])
        s=d['genres_list'].value_counts().head(12).sort_values()
        fig,ax=plt.subplots(figsize=(9,5)); s.plot(kind='barh',ax=ax,color='#4C78A8')
        ax.set(title='Most common genres',xlabel='Titles tagged with genre',ylabel='Genre')
    elif number == 14:
        cols=['release_year','runtime','seasons','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']
        corr=titles_df[cols].corr(numeric_only=True); fig,ax=plt.subplots(figsize=(9,7))
        sns.heatmap(corr,annot=True,fmt='.2f',cmap='vlag',center=0,vmin=-1,vmax=1,ax=ax)
        ax.set_title('Correlations among numeric title fields')
    elif number == 15:
        cols=['release_year','runtime','imdb_score','tmdb_score','type']; d=titles_df[cols].dropna()
        d=d.sample(min(700,len(d)),random_state=42)
        sns.pairplot(d,vars=cols[:-1],hue='type',corner=True,diag_kind='hist',plot_kws={'alpha':.35,'s':18})
        plt.suptitle('Pairwise patterns across a deterministic sample',y=1.02); plt.show(); return
    elif number == 16:
        d=titles_df[['production_countries_list']].explode('production_countries_list').dropna(subset=['production_countries_list'])
        s=d['production_countries_list'].value_counts().head(12).sort_values()
        fig,ax=plt.subplots(figsize=(9,5)); s.plot(kind='barh',ax=ax,color='#72B7B2')
        ax.set(title='Most represented production countries',xlabel='Titles tagged with country',ylabel='Country code')
    elif number == 17:
        d=titles_df[['production_countries_list','imdb_score']].explode('production_countries_list').dropna()
        s=d.groupby('production_countries_list')['imdb_score'].agg(['mean','count']).query('count >= 100').sort_values('mean').tail(12)
        fig,ax=plt.subplots(figsize=(9,5)); ax.barh(s.index,s['mean'],color='#54A24B')
        ax.set(title='Mean IMDb score by country (100+ rated titles)',xlabel='Mean IMDb score',ylabel='Country code',xlim=(0,10))
    elif number == 18:
        d=titles_df[['type','genres_list','imdb_score']].explode('genres_list').dropna()
        top=d['genres_list'].value_counts().head(10).index
        means=d[d['genres_list'].isin(top)].pivot_table(index='genres_list',columns='type',values='imdb_score',aggfunc='mean')
        fig,ax=plt.subplots(figsize=(8,6)); sns.heatmap(means,annot=True,fmt='.2f',cmap='YlGnBu',vmin=0,vmax=10,ax=ax)
        ax.set(title='Mean IMDb score by genre and format',xlabel='Format',ylabel='Genre')
    elif number == 19:
        d=titles_df.assign(age_certification=titles_df['age_certification'].fillna('Unknown')).groupby(['age_certification','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=d,x='age_certification',y='titles',hue='type',ax=ax)
        ax.tick_params(axis='x',rotation=45); ax.set(title='Certification labels by format',xlabel='Certification',ylabel='Titles')
    elif number == 20:
        d=credits_df.merge(titles_df[['id','imdb_score']],on='id',how='inner').dropna(subset=['imdb_score'])
        d=d.groupby(['role','name']).agg(title_count=('id','nunique'),mean_score=('imdb_score','mean')).reset_index()
        fig,axes=plt.subplots(1,2,figsize=(14,6))
        for ax,role in zip(axes,['ACTOR','DIRECTOR']):
            top=d.query('role == @role and title_count >= 10').nlargest(10,'mean_score').sort_values('mean_score')
            ax.barh(top['name'],top['mean_score'],color='#E45756')
            ax.set(title=f'Top rated {role.lower()} credits (10+ titles)',xlabel='Mean IMDb score',xlim=(0,10))
        axes[0].set_ylabel('Name'); axes[1].set_ylabel('')
    else:
        raise ValueError('Chart number must be from 1 through 20.')
    fig.tight_layout()
    plt.show()

plot_catalog_chart(1)

In [ ]:
# Chart - 20: Credited actors and directors with rated titles
plot_catalog_chart(20)

##### 1. Why did you pick the specific chart?

A bar chart compares the two format counts without implying a time trend.

##### 2. What is/are the insight(s) found from the chart?

Movies account for 8,514 of 9,871 rows, about 86%; shows account for 1,357. Catalog volume alone does not establish engagement.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use format mix to check whether acquisition planning matches supply. Budget decisions need viewing and cost data.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plot_catalog_chart(2)

##### 1. Why did you pick the specific chart?

A five-year binned histogram shows the long history and recent concentration with less year-level noise.

##### 2. What is/are the insight(s) found from the chart?

Releases span 1912–2022. The 2010s are the largest decade; the 2020s are incomplete in this file.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use historical mix for licensing research. Compare completed periods and track new years as the source updates.

#### Chart - 3

In [ ]:
plot_catalog_chart(3)

##### 1. Why did you pick the specific chart?

A box plot compares runtime spread and medians while hiding extreme points that would compress the distributions.

##### 2. What is/are the insight(s) found from the chart?

Movie runtime averages about 94 minutes, while show runtime averages about 36 minutes per episode. The units describe different formats.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use format-specific runtime expectations when scheduling or commissioning; show runtime is episode-level.

#### Chart - 4

In [ ]:
plot_catalog_chart(4)

##### 1. Why did you pick the specific chart?

A histogram with a density curve shows the shape, center, and tails of observed public scores.

##### 2. What is/are the insight(s) found from the chart?

There are 8,850 observed IMDb scores with mean about 5.98 and median 6.1; 1,021 titles have no score.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Ratings can shortlist titles, but show coverage and vote counts before investment decisions.

#### Chart - 5

In [ ]:
plot_catalog_chart(5)

##### 1. Why did you pick the specific chart?

A discrete distribution suits integer season counts and includes shows only.

##### 2. What is/are the insight(s) found from the chart?

Among 1,357 shows, the median is one season and the mean is about 2.79; a small long tail reaches 51 seasons.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Series depth can inform franchise planning. The long tail suggests a small set of long-running shows.

#### Chart - 6

In [ ]:
plot_catalog_chart(6)

##### 1. Why did you pick the specific chart?

A box plot compares score medians and spread between movie and show records.

##### 2. What is/are the insight(s) found from the chart?

Rated shows average about 7.12 and movies about 5.80. This does not show that format causes satisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The show mean supports a series-acquisition hypothesis to test, but cost and retention are required to establish value.

#### Chart - 7

In [ ]:
plot_catalog_chart(7)

##### 1. Why did you pick the specific chart?

A line chart shows how annual catalog volume changes and separates the formats.

##### 2. What is/are the insight(s) found from the chart?

Annual counts rise in later years. The source ends in 2022, so the latest apparent drop may reflect incomplete coverage.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use the trend to frame catalog-refresh questions. Do not read an incomplete decade as a completed slowdown.

#### Chart - 8

In [ ]:
plot_catalog_chart(8)

##### 1. Why did you pick the specific chart?

Grouped bars make decade-level format volume easy to compare.

##### 2. What is/are the insight(s) found from the chart?

The 2010s contribute 4,315 titles versus 1,189 in the 2000s. The 2020s count covers only three years.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The movie-heavy mix may motivate a controlled series test, but demand and production cost must be measured.

#### Chart - 9

In [ ]:
plot_catalog_chart(9)

##### 1. Why did you pick the specific chart?

A line chart summarizes average score by decade while retaining the numeric outcome.

##### 2. What is/are the insight(s) found from the chart?

Mean scores are about 6.06 for the 2010s and 5.77 for 2020–2022. Newer titles may have fewer votes or less mature ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

A small recent score difference supports monitoring, not a quality claim; check rating maturity and vote counts.

#### Chart - 10

In [ ]:
plot_catalog_chart(10)

##### 1. Why did you pick the specific chart?

A sampled scatter plot shows runtime-score association without overplotting every title.

##### 2. What is/are the insight(s) found from the chart?

Runtime has a weak negative correlation with IMDb score (about -0.10), so runtime alone is a poor quality proxy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Runtime can inform variety, but its weak relationship with score does not support selecting only shorter titles.

#### Chart - 11

In [ ]:
plot_catalog_chart(11)

##### 1. Why did you pick the specific chart?

A log-scaled scatter plot accommodates skewed vote counts and separates discussion volume from score.

##### 2. What is/are the insight(s) found from the chart?

IMDb votes and score have a weak positive correlation (about 0.17). Vote volume also reflects exposure and title age.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Treat votes as visibility and confidence context. Votes do not guarantee satisfaction.

#### Chart - 12

In [ ]:
plot_catalog_chart(12)

##### 1. Why did you pick the specific chart?

A log scale reveals patterns across a highly skewed popularity range.

##### 2. What is/are the insight(s) found from the chart?

TMDB popularity and IMDb score have very weak linear correlation (about 0.07), so visibility and rating are distinct signals.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Popularity can identify titles for discovery tests, but it is not a satisfaction target.

#### Chart - 13

In [ ]:
plot_catalog_chart(13)

##### 1. Why did you pick the specific chart?

A horizontal ranking keeps genre labels readable and shows overlapping catalog tags.

##### 2. What is/are the insight(s) found from the chart?

Drama, comedy, and thriller are common tags, appearing about 4,764, 2,987, and 2,119 times. A title may count in several genres.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Genre counts show crowded and underrepresented areas. Pair them with ratings, rights cost, and viewing data.

#### Chart - 14 - Correlation Heatmap

In [ ]:
plot_catalog_chart(14)

##### 1. Why did you pick the specific chart?

A heatmap summarizes pairwise linear association across numeric fields.

##### 2. What is/are the insight(s) found from the chart?

TMDB and IMDb scores have the strongest numeric association (about 0.58). Both are post-release signals.

Business impact: Use cross-platform ratings as a robustness check, not a replacement for Prime audience outcomes.

#### Chart - 15 - Pair Plot

In [ ]:
plot_catalog_chart(15)

##### 1. Why did you pick the specific chart?

A pair plot shows distributions and pairwise patterns together; a fixed sample keeps it readable.

##### 2. What is/are the insight(s) found from the chart?

The pair plot shows broad spread and limited linear relationships among most fields; the two public rating fields align more closely.

Business impact: Use the multivariate view to identify candidate segments for experiments. Association is not causation.

#### Chart - 16: Top production countries by title count

In [ ]:
plot_catalog_chart(16)

##### 1. Why did you pick the specific chart?

A horizontal ranking handles country codes and shows concentration across markets.

##### 2. What is/are the insight(s) found from the chart?

The US is tagged on 5,334 title rows, followed by India at 1,072 and the UK at 928. Country tags can include multiple markets.

##### 3. Will the gained insights help creating a positive business impact?

Use this distribution to review market coverage. It does not measure audience demand or acquisition cost.

#### Chart - 17: IMDb scores by production country

In [ ]:
plot_catalog_chart(17)

##### 1. Why did you pick the specific chart?

A minimum rated-title count reduces noisy rankings from small samples.

##### 2. What is/are the insight(s) found from the chart?

Among markets with at least 100 rated titles, Japan averages about 6.66, China 6.63, and the US 5.80. These are descriptive means.

##### 3. Will the gained insights help creating a positive business impact?

Use country differences to select markets for tests. Demand, rights, language, and cost data are needed before shifting spend.

#### Chart - 18: Genre ratings by format

In [ ]:
plot_catalog_chart(18)

##### 1. Why did you pick the specific chart?

A heatmap shows two categorical dimensions against a numeric outcome.

##### 2. What is/are the insight(s) found from the chart?

Genre-score patterns vary by format. Some cells have fewer observations, and titles can belong to multiple genres.

##### 3. Will the gained insights help creating a positive business impact?

Use format-specific comparisons to form hypotheses. Check group sizes and first-party engagement before acting.

#### Chart - 19: Certification coverage by format

In [ ]:
plot_catalog_chart(19)

##### 1. Why did you pick the specific chart?

Grouped bars show missing metadata and label mix by format.

##### 2. What is/are the insight(s) found from the chart?

Certification is missing for 6,487 of 9,871 rows, about 65.7%. R is the most common observed label, but Unknown is too large for a firm suitability conclusion.

##### 3. Will the gained insights help creating a positive business impact?

Improve certification coverage for discovery and family controls. The present labels do not describe the full catalog.

#### Chart - 20: Credited actors and directors with rated titles

In [ ]:
plot_catalog_chart(20)

##### 1. Why did you pick the specific chart?

Two panels compare roles and require ten credited titles to reduce one-title noise.

##### 2. What is/are the insight(s) found from the chart?

Among people meeting the threshold, Achyuth Kumar has a high actor mean and William Wyler a high director mean. Rankings can reflect genre and era.

##### 3. Will the gained insights help creating a positive business impact?

Use talent history to form a review shortlist. It does not show that a person caused ratings or future performance.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

Prioritize certification and rating-coverage improvements. Use format and genre results to frame acquisition experiments, and evaluate them against Prime-specific viewing hours, completion, renewal, and cost. Test country and talent hypotheses in matched title cohorts. Weight public ratings by votes and report rated-title coverage beside each average. Add rights cost, title-level consumption, release-window, and subscriber-retention data before estimating return on content investment.

# **Conclusion**

The catalog is weighted toward movies, while rated shows have a higher average IMDb score. Drama, comedy, and thriller are common tags, and documentation titles have a relatively high mean among sufficiently sized genre groups. IMDb and TMDB scores show some alignment, while runtime and popularity have weak linear relationships with IMDb score. Certification and rating gaps limit confidence. These findings support targeted tests and metadata cleanup. They do not establish that a genre, country, format, or credited person causes stronger Prime engagement or returns because the supplied data contain no viewing, cost, or retention outcomes.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***